In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# 1. Initialize API Client
_ = load_dotenv(find_dotenv())
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# 2. State Management
# This list holds the "memory" of the conversation
system_message = {'role': 'system', 'content': "You are a helpful assistant that explains things using Chinese idioms."}
context = [system_message]

# 3. Completion Function (Using the faster/cheaper gpt-4o-mini)
def get_completion(messages):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"⚠️ Error: {str(e)}"

# 4. UI Components
out = widgets.Output()
inp = widgets.Text(placeholder='Type your message...', layout=widgets.Layout(width='70%'))
button_chat = widgets.Button(description="Chat!", button_style='primary')
button_clear = widgets.Button(description="Clear Chat", button_style='warning')

# 5. Interaction Logic
def on_chat_clicked(b):
    user_input = inp.value
    if not user_input:
        return
    
    inp.value = '' # Clear input field immediately
    
    with out:
        # Display User message
        display(Markdown(f"**User**: {user_input}"))
        
        # Update context and get AI Response
        context.append({'role': 'user', 'content': user_input})
        response = get_completion(context)
        context.append({'role': 'assistant', 'content': response})
        
        # Display AI Response
        display(Markdown(f"**Assistant**: {response}"))
        display(Markdown("---"))

def on_clear_clicked(b):
    global context
    context = [system_message] # Reset memory to just the system prompt
    with out:
        clear_output() # Clear the visual chat history
        display(Markdown("*Chat history cleared.*"))

# Bind buttons to their functions
button_chat.on_click(on_chat_clicked)
button_clear.on_click(on_clear_clicked)

# 6. Render Layout
display(Markdown("### 🎋 Chinese Idiom Explainer"))
display(out)
display(widgets.HBox([inp, button_chat, button_clear]))


### 🎋 Chinese Idiom Explainer

Output()